# Sesión 7 · Combinación de filas de datos con Pandas

**Introducción a la Ingeniería en Ciencia de Datos — período 2026-3C**
Universidad del Magdalena · Facultad de Ingeniería

- Ing. Alexander Bustamante Martínez, MSc., PhD. — abustamante@unimagdalena.edu.co
- Ing. Andrés Eduardo Paternina Ariza, Esp. — apaternina@unimagdalena.edu.co

---

### ¿Qué vas a aprender hoy?

Al terminar este notebook serás capaz de:

1. Reconocer la **clave primaria (PK)** y la **clave foránea (FK)** de una tabla.
2. Leer cómo se **relacionan** varias tablas (relación **1 a N**).
3. Combinar tablas con `pd.merge()` usando `on`, o `left_on` y `right_on`.
4. Elegir el **tipo de combinación** con `how`: `inner`, `left`, `right` u `outer`.
5. **Verificar** el número de filas antes y después de cada combinación.
6. Manejar **columnas repetidas** con los sufijos `_x` / `_y` y el parámetro `suffixes`.
7. Combinar tablas y luego **agrupar, filtrar y ordenar** el resultado.

> **Cómo usar este notebook:** léelo de arriba hacia abajo y ejecuta cada celda de código en orden con `Shift + Enter`. En cada pregunta, escribe tu solución en la celda vacía siguiendo la explicación de clase.

---
## 1. De un archivo a varias tablas

En clases anteriores trabajamos con **un único archivo CSV** que contenía todos los datos de los futbolistas del club Promesas Bananeras. En la vida real, los datos se organizan en **múltiples archivos o tablas** relacionadas entre sí a través de **columnas clave**.

Para esta clase contamos con **cuatro archivos CSV**:

| Archivo CSV | Filas | ¿Qué contiene? | Clave primaria (PK) | Claves foráneas (FK) |
|---|---|---|---|---|
| `pb_players.csv` | 100 | Datos personales de cada jugador | `player_id` | `birth_country_id`, `represented_country_id` |
| `pb_clubs.csv` | 36 | Clubes del universo de datos | `club_id` | `country_id` |
| `pb_countries.csv` | 16 | Catálogo de países | `country_id` | No tiene |
| `pb_goals_scored.csv` | 450 | Cada gol marcado por un jugador de Promesas Bananeras | `goal_id` | `player_id`, `club_id`, `opponent_club_id` |

- **Clave primaria (PK):** columna que identifica de forma **única** cada fila de una tabla.
- **Clave foránea (FK):** columna que **referencia la PK de otra tabla**, creando así la relación entre ambas.

### 1.1 Preparar el entorno en Google Colab

1. Entra a <https://colab.google/> con tu cuenta de Google.
2. Descarga del **campus virtual** los **cuatro** archivos CSV.
3. Súbelos a `/content`: en el panel izquierdo, ícono de **carpeta 📁 → Subir**.

> ⚠️ Los archivos subidos a `/content` se **borran** al cerrar la sesión de Colab. Si vuelves otro día, súbelos de nuevo.

In [ ]:
import pandas as pd
import numpy as np

# Carpeta de los archivos en Colab. Si trabajas en tu computador (Jupyter local),
# cambia la carpeta por "" (cadena vacía) si los CSV están junto al notebook.
CARPETA = "/content/"

df_players   = pd.read_csv(CARPETA + "pb_players.csv")
df_clubs     = pd.read_csv(CARPETA + "pb_clubs.csv")
df_countries = pd.read_csv(CARPETA + "pb_countries.csv")
df_goals     = pd.read_csv(CARPETA + "pb_goals_scored.csv")

print("df_players:  ", df_players.shape)     # (100, 9)
print("df_clubs:    ", df_clubs.shape)       # (36, 4)
print("df_countries:", df_countries.shape)   # (16, 2)
print("df_goals:    ", df_goals.shape)       # (450, 6)

---
## 2. Cómo se relacionan las tablas

```
pb_countries (16)                                pb_clubs (36)
┌─────────────────┐                              ┌─────────────────┐
│ country_id   PK │ 1 ──────────────────────── N │ club_id      PK │
│ country_name    │                              │ club_name       │
└─────────────────┘                              │ country_id   FK │
    1        1                                   │ city            │
    │        │                                   └─────────────────┘
    N        N                                        1        1
┌──────────────────────────┐                          │        │
│ pb_players (100)         │                          N        N
│ player_id            PK  │ 1 ──── N ┌──────────────────────────┐
│ full_name                │          │ pb_goals_scored (450)    │
│ birth_date               │          │ goal_id              PK  │
│ position                 │          │ player_id            FK  │
│ dominant_foot            │          │ club_id              FK  │
│ height_cm                │          │ opponent_club_id     FK  │
│ weight_kg                │          │ minute                   │
│ birth_country_id     FK  │          │ goal_type                │
│ represented_country_id FK│          └──────────────────────────┘
└──────────────────────────┘
```

- **PK** = identifica cada fila · **FK** = apunta a la PK de otra tabla · **1 ── N** = una fila se relaciona con muchas.

> 📌 **Dato clave:** todos los goles son de Promesas Bananeras: `club_id` siempre vale `PB001`. El **rival** está en `opponent_club_id`.

### 2.1 La clave primaria (PK)

Una PK:
- **nunca se repite**,
- **nunca está vacía**,
- hay **una por tabla**.

| Tabla | PK |
|---|---|
| `pb_players` | `player_id` |
| `pb_clubs` | `club_id` |
| `pb_countries` | `country_id` |
| `pb_goals_scored` | `goal_id` |

> 💬 **¿Por qué importa?** Si dos jugadores tuvieran el mismo `player_id`, no sabríamos a quién pertenece cada gol.

### 2.2 La clave foránea (FK)

- Una FK **sí puede repetirse**: un mismo `player_id` aparece muchas veces en `pb_goals_scored`.
- Cada valor de una FK **debe existir como PK** en la tabla a la que apunta.
- La pareja **FK = PK** es la condición que le daremos a `merge`.

### 2.3 La relación 1 a N

Un jugador (lado **1**, `pb_players`) puede marcar muchos goles (lado **N**, `pb_goals_scored`).

> 📌 Al combinar las dos tablas, **los datos del jugador se repiten en cada uno de sus goles**. Por eso el resultado tiene una fila por gol, no una por jugador.

### 2.4 Dos FK que apuntan a la misma tabla

**`pb_players` → `pb_countries`**

| FK | ¿Qué significa? |
|---|---|
| `birth_country_id` | País donde nació |
| `represented_country_id` | País que representa |

**`pb_goals_scored` → `pb_clubs`**

| FK | ¿Qué significa? |
|---|---|
| `club_id` | Club del goleador (siempre `PB001`) |
| `opponent_club_id` | Club rival del partido |

> ⚠️ **La pregunta decide qué FK usar:** «país de nacimiento» → `birth_country_id`; «país que representa» → `represented_country_id`.
> Aquí la FK y la PK tienen **nombres distintos** (`birth_country_id` ≠ `country_id`). Por eso usaremos `left_on` y `right_on`.

---
## 3. `pd.merge()`: combinar tablas

**`merge`** es la operación que utiliza Pandas para combinar conjuntos de datos con base en la **condición de combinación**, alineando filas cuando los valores de la condición coinciden.

### 3.1 Anatomía de `pd.merge`: cuatro decisiones

```python
resultado = pd.merge(
    df_izquierda,          # 1. tabla izquierda (left)
    df_derecha,            # 2. tabla derecha (right)
    on="columna_clave",    # 3. condición de combinación
    how="inner"            # 4. tipo de combinación
)
```

| # | Decisión | Qué significa |
|---|---|---|
| 1 | **Tabla izquierda** | La primera tabla que escribes. Sus columnas quedan a la izquierda del resultado. |
| 2 | **Tabla derecha** | La segunda tabla. Sus columnas se agregan a la derecha. |
| 3 | **`on` / `left_on` + `right_on`** | Columnas cuyos valores deben coincidir (FK = PK). Mismo nombre: `on=`. Nombres distintos: `left_on=` y `right_on=`. |
| 4 | **`how`** | `inner`, `left`, `right` u `outer`. Decide qué filas se conservan. Si no lo escribes, Pandas usa `inner`. |

### 3.2 El parámetro `how` decide qué filas se conservan

| `how` | ¿Qué filas conserva? | Si una fila no tiene pareja… |
|---|---|---|
| `inner` | Solo las que coinciden en las dos tablas | Se elimina |
| `left` | Todas las de la tabla izquierda | Columnas de la derecha → `NaN` |
| `right` | Todas las de la tabla derecha | Columnas de la izquierda → `NaN` |
| `outer` | Todas las de ambas tablas | `NaN` en el lado que falte |

> ⚠️ Si no escribes `how`, Pandas usa `inner` y **puede eliminar filas sin avisarte**.
> 🛠️ Usa `left` cuando la pregunta dice **«todos»** o **«incluso los que no tienen…»**.

### 3.3 Verifica el número de filas antes y después de cada merge

Las combinaciones requieren que especifiquemos cómo emparejar los datos. Dependiendo del tipo de combinación que elijamos, **es posible que perdamos algunos datos**. Por eso es importante revisar el conteo de filas **antes y después** de la combinación.

| Si obtienes… | Probablemente… |
|---|---|
| **Menos** filas de las esperadas | Un `inner` eliminó filas sin pareja. |
| **Más** filas de las esperadas | La relación es 1 a N y una fila se repitió. |

> 🛠️ Usa `len(df)` o `df.shape` antes y después de cada `merge`.

### 3.4 `on` o `left_on` / `right_on`: depende del nombre de la columna

| Mismo nombre → `on` | Nombres distintos → `left_on` y `right_on` |
|---|---|
| `country_id` existe en las dos tablas | La FK y la PK no se llaman igual |
| `pd.merge(df_clubs, df_countries, on="country_id")` | `pd.merge(df_players, df_countries, left_on="birth_country_id", right_on="country_id")` |

> 📌 `left_on` nombra la columna de la tabla **izquierda**; `right_on`, la de la tabla **derecha**. Si inviertes el orden de las tablas, inviertes también estos parámetros.
> Con `left_on`/`right_on` el resultado **conserva las dos columnas** (`birth_country_id` y `country_id`).

### 3.5 Columnas repetidas reciben los sufijos `_x` y `_y`

Cuando las dos tablas tienen una columna con el **mismo nombre** que no se usó como `on`, Pandas le agrega sufijos:

- `_x` = columna de la tabla **izquierda**.
- `_y` = columna de la tabla **derecha**.
- Puedes poner nombres más claros: `suffixes=("_goleador", "_rival")`.

> 🛠️ Antes de usar una columna, revisa `df.columns` para saber cómo se llama **después** del merge.

### 3.6 Después de un `left`: `count` ignora los `NaN`, `size` no

Cuando un `left` conserva una fila **sin pareja**, sus columnas de la derecha quedan vacías. Al agrupar:

| Función | Qué cuenta |
|---|---|
| `count` sobre una columna de la tabla derecha | Solo valores **no vacíos** |
| `size` | **Filas**, aunque tengan `NaN` |

> ⚠️ Para contar elementos de la tabla derecha después de un `left`, cuenta **una columna de esa tabla** con `"count"`. `size` daría un conteo falso: un **error silencioso** que el código no te avisa.

---
## 4. Preguntas de clase

Para cada pregunta, antes de escribir código responde: **¿qué tablas necesito? → ¿cuál es la condición de combinación (FK = PK)? → ¿qué tipo de `how`? → ¿qué hago después (renombrar, filtrar, agrupar, ordenar)?**

### ❓ Pregunta 1: ¿Cuál es el nombre y país de nacimiento de cada jugador?

In [ ]:
# Escribe aquí tu solución a la Pregunta 1

### ❓ Pregunta 2: ¿Cuál es el nombre del club, el país y la ciudad de todos los clubes ubicados en Colombia?

In [ ]:
# Escribe aquí tu solución a la Pregunta 2

### ❓ Pregunta 3: ¿Cuál es la cantidad de clubes que tiene cada país?

*Mostrar los resultados ordenados por la cantidad de clubes (de mayor a menor). Para los países que no registren clubes, mostrar 0 en la cantidad de clubes. Mostrar los países que tienen menos de tres clubes.*

In [ ]:
# Escribe aquí tu solución a la Pregunta 3

### ❓ Pregunta 4: ¿Cuántos goles ha marcado cada jugador?

*Mostrar el nombre del jugador y la cantidad de goles, ordenados de mayor a menor.*

In [ ]:
# Escribe aquí tu solución a la Pregunta 4

### ❓ Pregunta 5: ¿Cuántos goles ha marcado Promesas Bananeras contra clubes de cada país?

*Mostrar el país del club rival y la cantidad de goles, ordenados de mayor a menor.*

In [ ]:
# Escribe aquí tu solución a la Pregunta 5

---
## 5. Taller en grupos

Reúnanse en **grupos de tres** y resuelvan las siguientes necesidades de datos. Escriban su solución en la celda vacía de cada ejercicio.

### 🧩 Ejercicio 1

**¿Cuál es el nombre y país que representa cada jugador?**

In [ ]:
# Escribe aquí tu solución al Ejercicio 1

### 🧩 Ejercicio 2

**¿Cuántos goles han marcado los jugadores de cada posición?**
Mostrar la posición y la cantidad de goles, ordenados de mayor a menor.

In [ ]:
# Escribe aquí tu solución al Ejercicio 2

### 🧩 Ejercicio 3

**¿Qué jugadores no registran goles?**
*Pista: combina con `how="left"` y conserva las filas donde `goal_id` es `NaN` (usa `.isna()`).*

In [ ]:
# Escribe aquí tu solución al Ejercicio 3

### 🧩 Ejercicio 4

**¿Qué jugadores marcaron goles de cabeza después del minuto 80 y contra qué club?**
Mostrar el nombre del jugador, el nombre del club rival y el minuto del gol.

In [ ]:
# Escribe aquí tu solución al Ejercicio 4

---
## 6. Resumen: lo que debes recordar

| # | Idea | Para recordar |
|---|---|---|
| 1 | **PK y FK** | La PK identifica cada fila y no se repite. La FK apunta a la PK de otra tabla y sí puede repetirse. |
| 2 | **Condición de combinación** | `merge` une filas cuando FK = PK. Mismo nombre: `on`. Nombres distintos: `left_on` y `right_on`. |
| 3 | **El parámetro `how`** | `inner` solo coincidencias; `left` / `right` conservan un lado; `outer` todo. Si no lo escribes: `inner`. |
| 4 | **Cuenta las filas** | Usa `len(df)` antes y después. Menos filas: se perdieron datos. Más filas: relación 1 a N. |
| 5 | **`NaN` después de un `left`** | Las filas sin pareja quedan en `NaN`. Para contarlas bien usa `"count"` sobre una columna de la tabla derecha, no `size`. |
| 6 | **Columnas repetidas** | Reciben `_x` (izquierda) y `_y` (derecha). Usa `suffixes` para darles nombres claros. |